## Deploying a MRI classification app using Huggingface and Gradio

1. Create a demo Gradio app that uploads and displays an input image
   * Install Gradio
   * Define a function to create a gallery of 3-5 images. It should be possible to upload the images through the interface
   * Add UI elements like a text description of the app and its function.

2.  App Component 1: Create a Gradio interface for MRI classification
   * Load a vision transformer model trained for MRI classification (into tumor and no tumor classes).
   * Load image_processors for preprocessing input images.
   * Write a function to predict image class using the VIT model on an input image.
   * Write a Gradio interface function that allows a user to upload an input image and launches the prediction function on the imag.
   * Include components in the Gradio function to display the VIT classification result on the interface (plot or print probabilities for tumor and no tumor classes).

3. App Component 2:Create a Gradio interface for MRI segmentaion
   * Load a transformer model trained for MRI segmentation (e.g SegFormer or Maskformer).
   * Load image_processors for preprocessing input images for segmentation.
   * Write a function to predict image segmentation using the segmentation model on an input image.
   * Write a Gradio interface function that allows a user to upload an input image and launches the prediction function on the image.
   * Include components in the Gradio function to display the segmentation result on the interface (segmented tumor image).

4. Assemble all contents in single Gradio interface
   * Assemble app contents - text section, galley section and image analysis sections
   * Create a tabbed interface for classification and segmentations sections
   * Upload on Huggingface space


In [ ]:
pip install gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers

In [ ]:

import random
import glob

import gradio as gr



path_im = '/content/drive/MyDrive/colab_notebooks/lp2_files/mri_data_seg/images/train'

def select_rand():
    images = [
        (random.choice(glob.glob(path_im+ '/*.png')), f"label {i}" if i != 0 else "label" * 50)
        for i in range(4)
    ]
    return images


with gr.Blocks() as demo:
    gr.Markdown(
    """
    # Welcome to the MRI analysis app
    You can run Transformer models in this app to classify and segment MR data
    """)
    with gr.Column(variant="panel"):
        with gr.Row():

            btn = gr.Button("Display MR images", scale=0)

        gallery = gr.Gallery(
            label="Generated images", show_label=False, elem_id="gallery"
        , columns=[2], rows=[2], object_fit="contain", height="auto")

    btn.click(select_rand, None, gallery)

if __name__ == "__main__":

  demo.launch(debug=True)


## App Component 1: Create a Gradio app to upload a MRI image and run VIT classification on it for (tumor/no-tumor)

In [ ]:
!pip install transformers

In [ ]:
from huggingface_hub import login
login(token='YOUR_HF_TOKEN_HERE')

In [ ]:
from transformers import AutoImageProcessor
from transformers import TFAutoModelForImageClassification

model = TFAutoModelForImageClassification.from_pretrained("akar49/mri_classifier")

In [ ]:

import tensorflow as tf
def predict(inp):

  image_processor = AutoImageProcessor.from_pretrained("akar49/mri_classifier")
  inputs = image_processor(inp, return_tensors="tf")

  logits = model(inputs).logits
  pred = int(tf.math.argmax(logits, axis=-1)[0])
  return {'tumor': str(pred), 'no-tumor': str(1-pred)}


In [ ]:
import gradio as gr
gr.Interface(fn=predict,
             inputs=gr.Image(shape=(224, 224)),
             outputs="label").launch(debug=True)

## App Component 2: Create a Gradio app for segmenting tumor region

In [ ]:
from transformers import MaskFormerForInstanceSegmentation

from PIL import Image

model = MaskFormerForInstanceSegmentation.from_pretrained("akar49/Maskformer-MRIseg_model")


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
from transformers import MaskFormerImageProcessor

preprocessor = MaskFormerImageProcessor(ignore_index=255, reduce_labels=False, do_resize=False, do_rescale=False, do_normalize=False)

In [ ]:
def run_inference(image):
  model = MaskFormerForInstanceSegmentation.from_pretrained("akar49/Maskformer-MRIseg_model-Sep")
  preprocessor = MaskFormerImageProcessor(ignore_index=255, reduce_labels=False, do_resize=False, do_rescale=False, do_normalize=False)
  image = np.array(image)
  image= np.float32(image)
  image = cv2.resize(image, (224, 224),
               interpolation = cv2.INTER_LINEAR)
  image = image.transpose(2,1,0)
  model = model.to(device)
  model.eval()
  image_tensor = torch.tensor(image)
  image_tensor = image_tensor.unsqueeze(0).to(device)
  with torch.no_grad():
    outputs = model(image_tensor)
    predictions = preprocessor.post_process_semantic_segmentation(outputs)
    probs = [t.detach().numpy() for t in predictions]
    prediction= Image.fromarray((probs[0] * 255).astype(np.uint8))
    prediction= prediction.resize((300,300), Image.Resampling.LANCZOS)
    return prediction

In [ ]:
import gradio as gr
import numpy as np
import cv2

iface = gr.Interface(run_inference, gr.inputs.Image(shape=(224, 224)),"image").launch(debug=True)

## Put components together in one app

In [ ]:
## Put all app components together

import random
import glob
import gradio as gr
import tensorflow as tf
import numpy as np
import cv2
from PIL import Image
import torch

from transformers import AutoImageProcessor
from transformers import TFAutoModelForImageClassification
from transformers import MaskFormerForInstanceSegmentation
from transformers import MaskFormerImageProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
## Huggingface login


## MRI Display Items
path_im = '/content/drive/MyDrive/colab_notebooks/lp2_files/mri_data_seg/images/train'
def select_rand():
    images = [
        (random.choice(glob.glob(path_im+ '/*.png')), f"label {i}" if i != 0 else "label" * 50)
        for i in range(4)
    ]
    return images

## MRI Classification items
model = TFAutoModelForImageClassification.from_pretrained("akar49/mri_classifier")
def predict(inp):

  image_processor = AutoImageProcessor.from_pretrained("akar49/mri_classifier")
  inputs = image_processor(inp, return_tensors="tf")

  logits = model(inputs).logits
  pred = int(tf.math.argmax(logits, axis=-1)[0])
  return {'tumor': str(pred), 'no-tumor': str(1-pred)}


# MRI Segmentation items

def run_inference(image):
  model = MaskFormerForInstanceSegmentation.from_pretrained("akar49/Maskformer-MRIseg_model-Sep")
  preprocessor = MaskFormerImageProcessor(ignore_index=255, reduce_labels=False, do_resize=False, do_rescale=False, do_normalize=False)

  image = np.array(image)
  image= np.float32(image)
  image = cv2.resize(image, (224, 224),
               interpolation = cv2.INTER_LINEAR)
  image = image.transpose(2,1,0)
  model = model.to(device)
  model.eval()
  image_tensor = torch.tensor(image)
  image_tensor = image_tensor.unsqueeze(0).to(device)
  with torch.no_grad():
    outputs = model(image_tensor)
    predictions = preprocessor.post_process_semantic_segmentation(outputs)
    probs = [t.detach().cpu().numpy() for t in predictions]
    prediction= Image.fromarray((probs[0] * 255).astype(np.uint8))
    prediction= prediction.resize((300,300), Image.Resampling.LANCZOS)
    return prediction

## APP

with gr.Blocks() as demo:
    gr.Markdown(
    """
    # Welcome to the MRI analysis app
    You can run Transformer models in this app to classify and segment MR data

    ### First display some MR images

    """)
    with gr.Column(variant="panel"):
        with gr.Row():

            btn = gr.Button("Display MR images", scale=0)

        gallery = gr.Gallery(
            label="Generated images", show_label=False, elem_id="gallery"
        , columns=[2], rows=[2], object_fit="contain", height="auto")

    btn.click(select_rand, None, gallery)

    with gr.Tab("MRI Classification"):
      gr.Interface(fn=predict,
             inputs=gr.Image(shape=(224, 224)),
             outputs="label")

    with gr.Tab("MRI Segmentation"):

       gr.Interface(fn=run_inference, inputs=inputImage, outputs=outputImage)



if __name__ == "__main__":
    demo.launch(debug=True)